# Production plan with a central warehouse

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/central_warehouse.ipynb)

A company assembles electric bicycles in three plants and plans how many bikes each plant assembles each month from January to June. In the second version of the model, M1, each plant holds stock from one month to the next. With forecast 3, which has a peak in February, M1 has no solution: the plants cannot assemble enough bikes in advance. The orders of February can still be delivered in March, as backorders, and M1 does not represent backorders. This notebook uses the third version, M2: the plants ship to a central warehouse, which holds stock, delivers the orders and backorders the demand it cannot deliver in its month. The plants and the warehouse have a storage capacity.

Contents of the notebook:

1. Data of the plants and the warehouse, and forecast 3.
2. Linear model with a central warehouse.
3. Optimal plan under forecast 3.
4. Four tests, each with one datum changed: the limit on the backorders at the end of June, the storage capacity of the plants, the backorder cost and the storage capacity of the warehouse.

The cells run in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders and tick boxes): after changing a value, run the cell again. Outside Colab the form is a line such as `backorder_cost = 60`: edit the value and run the cell.

The table gives the symbols of M0 to M2. The column «From» gives the version where each symbol first appears. The warehouse is written $w$.

| Symbol | Meaning | Unit | From |
|---|---|---|---|
| $F$, $f$ | Plants | — | M0 |
| $T$, $t$ | Months, January to June; $t = 0$ is the end of December | — | M0 |
| $\tau$ | Last month of the horizon | — | M1 |
| $w$ | The central warehouse | — | M2 |
| $c_f$ | Unit assembly cost at plant $f$ | €/bike | M0 |
| $K_{ft}$ | Assembly capacity of plant $f$ in month $t$ | bikes/month | M0 |
| $d_t$ | Orders received in month $t$ | bikes/month | M0 |
| $c^h_f$; $c^h_w$ | Holding cost at plant $f$; at the warehouse | €/(bike·month) | M1; M2 |
| $h_{f0}$; $h_{w0}$ | Initial stock at plant $f$; at the warehouse, at the end of December | bikes | M1; M2 |
| $h^{\text{end}}_f$; $h^{\text{end}}_w$ | Required final stock at plant $f$; at the warehouse | bikes | M1; M2 |
| $K^h_f$; $K^h_w$ | Storage capacity of plant $f$; of the warehouse | bikes | M2 |
| $c^b$ | Backorder cost: one bike backordered for one month | €/(bike·month) | M2 |
| $b_0$ | Backorders at the end of December | bikes | M2 |
| $\bar b$ | Maximum backorders at the end of the horizon | bikes | M2 |
| $p_{ft}$ | Bikes assembled at plant $f$ in month $t$ | bikes/month | M0 |
| $h_{ft}$; $h_{wt}$ | Bikes in stock at plant $f$; at the warehouse, at the end of month $t$ | bikes | M1; M2 |
| $x_{ft}$ | Bikes shipped from plant $f$ in month $t$, to the warehouse in M2 | bikes/month | M1 |
| $s_t$ | Deliveries: bikes the warehouse delivers in month $t$ to orders of that month or to backorders | bikes/month | M2 |
| $b_t$ | Backorders at the end of month $t$: orders received and not yet delivered | bikes | M2 |

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/central_warehouse.md) explains all the data and results. The data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import central_warehouse as cw

## 1. Data of the plants and the warehouse, and forecast 3

The first table gives, for each plant, the data of M1 and the storage capacity that M2 adds. Holding a bike at a plant costs less than at the warehouse, but a plant has little storage capacity, because space at a plant is expensive. The second table gives the data of the central warehouse and of the backorders: the backorder cost, the backorders at the end of December and the maximum backorders at the end of June (none). The third table is forecast 3: the orders the warehouse receives each month, with a peak in February.

In [ ]:
# @title Data of the plants and the warehouse, and forecast 3 { display-mode: "form" }
cw.show_data()

## 2. Linear model with a central warehouse

$$
\begin{aligned}
\min \quad & \sum_{t \in T} \Big( \sum_{f \in F} \left( c_f\, p_{ft} + c^h_f\, h_{ft} \right) + c^h_w\, h_{wt} + c^b\, b_t \Big) \\
\text{s.t.} \quad & h_{f,t-1} + p_{ft} = x_{ft} + h_{ft} && \forall f \in F,\ t \in T && \text{(stock balance at each plant)} \\
& x_{ft} \le h_{f,t-1} + p_{ft} && \forall f \in F,\ t \in T && \text{(shipment limit at each plant)} \\
& h_{w,t-1} + \sum_{f \in F} x_{ft} = s_t + h_{wt} && \forall t \in T && \text{(stock balance at the warehouse)} \\
& s_t \le h_{w,t-1} + \sum_{f \in F} x_{ft} && \forall t \in T && \text{(shipment limit at the warehouse)} \\
& b_{t-1} + d_t = s_t + b_t && \forall t \in T && \text{(backorder balance)} \\
& p_{ft} \le K_{ft},\quad h_{ft} \le K^h_f && \forall f \in F,\ t \in T && \text{(capacity; storage capacity of the plants)} \\
& h_{wt} \le K^h_w && \forall t \in T && \text{(storage capacity of the warehouse)} \\
& b_{\tau} \le \bar b && && \text{(backorders at the end)} \\
& h_{f\tau} \ge h^{\text{end}}_f \quad \forall f \in F, \qquad h_{w\tau} \ge h^{\text{end}}_w && && \text{(required final stock)} \\
& p_{ft},\ x_{ft},\ h_{ft},\ h_{wt},\ s_t,\ b_t \ge 0 && \forall f \in F,\ t \in T
\end{aligned}
$$

with $h_{f0}$, $h_{w0}$ and $b_0$ given. The plants ship to the warehouse, which has two balances. In the stock balance, what the warehouse holds at the start of the month plus what arrives equals its deliveries plus what it holds at the end. In the backorder balance, the backorders at the start of the month plus the orders of the month equal the deliveries plus the backorders at the end. The deliveries, $s_t$, are in both balances: each bike delivered leaves the stock and meets an order or a backorder.

## 3. Optimal plan under forecast 3

The cell first gives the reason why M1 has no solution under forecast 3. Then it solves M2 and gives, for each month, the bikes assembled at each plant, shipped to the warehouse and in stock at each plant; the stock of the warehouse, its deliveries and the backorders at the end of the month; and the cost of the plan. Below, two tables of **shadow prices** (€/bike): what one more order each month would cost, and the change in cost with one more bike of storage capacity at each site (negative: a saving). The figure on the left stacks the bikes assembled by each plant against the capacity of the plants, the orders and the deliveries; the one on the right stacks the stock at the plants and at the warehouse, with the backorders beside it.

Where is the seasonal stock assembled in January held, and in what order are the plants and the warehouse filled? Why are some orders of February backordered until March, and not delivered earlier? What would one more bike of storage capacity at the warehouse in January be worth?

In [ ]:
# @title Optimal plan under forecast 3 { display-mode: "form" }
cw.warehouse_plan();

## 4. Backorders at the end of June

Demand can be backordered, but no order may be backordered at the end of June. The tick box removes that limit. The model does not represent the months after June: in it, an order that is never delivered costs only the backorder cost of the months until the end of June, 60 € each.

Run the cell without the limit. How many bikes do the plants assemble, and how many orders are backordered at the end of June? Why is that plan cheaper, and why would the company never accept it?

In [ ]:
# @title Optimal plan with or without the limit on the backorders at the end { display-mode: "form" }
limit_backorders_at_the_end = True  # @param {type:"boolean"}
cw.backorders_at_the_end_test(limit_backorders_at_the_end);

## 5. Storage capacity of the plants

The slider sets the storage capacity of plant 1. Those of plants 2 and 3 move with it, 10 and 20 bikes lower, so that each plant can still hold its initial stock. A plant is the cheapest place to hold a bike but has little storage capacity; the seasonal stock that does not fit goes to the warehouse, whose holding cost is higher and whose storage capacity is also limited.

Raise the storage capacity of the plants, then lower it. Where is the seasonal stock of January held, and how many orders of February are backordered? Does one more bike of storage capacity at a plant save the same as one at the warehouse?

In [ ]:
# @title Optimal plan for a storage capacity of the plants { display-mode: "form" }
storage_capacity_plant_1 = 40  # @param {type:"slider", min:20, max:200, step:10}
cw.plant_storage_test(storage_capacity_plant_1);

## 6. Backorder cost under forecast 2

This cell uses forecast 2 of M0 and M1, with its peak in April and May, and changes the backorder cost. A backorder is one more way to meet a peak, besides seasonal stock and the most expensive plant, and its cost decides whether the optimal plan has backorders.

Lower the backorder cost from 60. Below what cost is demand backordered instead of assembled at plant 3, and which orders? What happens at exactly that cost?

In [ ]:
# @title Optimal plan for a backorder cost { display-mode: "form" }
backorder_cost = 60  # @param {type:"slider", min:0, max:100, step:5}
cw.backorder_cost_test(backorder_cost);

## 7. Storage capacity of the warehouse

The slider sets the storage capacity of the warehouse, with forecast 3. The bikes assembled in advance are held somewhere until the orders arrive, and a small warehouse limits the seasonal stock.

Lower the storage capacity of the warehouse, then raise it. How many orders of February are backordered in each case? From what storage capacity does more room change nothing, and why are some orders still backordered?

In [ ]:
# @title Optimal plan for a storage capacity of the warehouse { display-mode: "form" }
warehouse_storage_capacity = 150  # @param {type:"slider", min:0, max:300, step:10}
cw.warehouse_storage_test(warehouse_storage_capacity);

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/central_warehouse.md) explains all the numbers and has a script that prints them.